In [30]:
from collections import deque
import pandas as pd
import os

In [31]:
project_dir = os.path.dirname(os.getcwd())
games = pd.read_csv(os.path.join(project_dir, "data/cleaned_chess.csv"))
games["Date"] = pd.to_datetime(games["Date"])

In [32]:
games

,Site,Date,White,Black,Result,WhiteElo,BlackElo,ECO
0,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Farfan Ortiz,V",0.5,2137.0,2206.0,D13
1,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Salinas Herrera,P",0.0,2137.0,2409.0,A46
2,Talcahuano CHI,2012-06-02,"Farfan Ortiz,E","Vizama Riveros,C",0.0,2236.0,2256.0,C42
3,Talcahuano CHI,2012-06-02,"Rojas Sepulveda,E","Diaz Vasquez,M",0.5,2315.0,2212.0,A46
4,Talcahuano CHI,2012-06-02,"Estay Torreblanca,O","Alarcon,Rod",1.0,2214.0,2174.0,C17
...,...,...,...,...,...,...,...,...
3255651,Aktobe KAZ,2026-06-08,"Kabinazar,Nurmukhammed","Tarhan,Adar",0.5,2312.0,2482.0,B11
3255652,Aktobe KAZ,2026-06-08,"Zhalmakhanov,Ramazan","Kornienko,Konstantin",0.0,2478.0,2388.0,B28
3255653,Aktobe KAZ,2026-06-08,"Dronavalli,Harika","Degenbaev,Aziz",0.5,2466.0,2320.0,A06
3255654,Aktobe KAZ,2026-06-08,"Rozum,I","Makhnev,Denis",1.0,2455.0,2551.0,A40


Elo 

In [33]:
games["EloDiff"] = games["WhiteElo"] - games["BlackElo"]
games["AvgElo"] = (games["WhiteElo"] + games["BlackElo"]) / 2
games["HigherRatedPlayer"] = games.loc[:,["WhiteElo", "BlackElo"]].idxmax(axis="columns").replace("BlackElo", 0).replace("WhiteElo", 1)
games

,Site,Date,White,Black,Result,WhiteElo,BlackElo,ECO,EloDiff,AvgElo,HigherRatedPlayer
0,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Farfan Ortiz,V",0.5,2137.0,2206.0,D13,-69.0,2171.5,0
1,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Salinas Herrera,P",0.0,2137.0,2409.0,A46,-272.0,2273.0,0
2,Talcahuano CHI,2012-06-02,"Farfan Ortiz,E","Vizama Riveros,C",0.0,2236.0,2256.0,C42,-20.0,2246.0,0
3,Talcahuano CHI,2012-06-02,"Rojas Sepulveda,E","Diaz Vasquez,M",0.5,2315.0,2212.0,A46,103.0,2263.5,1
4,Talcahuano CHI,2012-06-02,"Estay Torreblanca,O","Alarcon,Rod",1.0,2214.0,2174.0,C17,40.0,2194.0,1
...,...,...,...,...,...,...,...,...,...,...,...
3255651,Aktobe KAZ,2026-06-08,"Kabinazar,Nurmukhammed","Tarhan,Adar",0.5,2312.0,2482.0,B11,-170.0,2397.0,0
3255652,Aktobe KAZ,2026-06-08,"Zhalmakhanov,Ramazan","Kornienko,Konstantin",0.0,2478.0,2388.0,B28,90.0,2433.0,1
3255653,Aktobe KAZ,2026-06-08,"Dronavalli,Harika","Degenbaev,Aziz",0.5,2466.0,2320.0,A06,146.0,2393.0,1
3255654,Aktobe KAZ,2026-06-08,"Rozum,I","Makhnev,Denis",1.0,2455.0,2551.0,A40,-96.0,2503.0,0


<h>Player History</h>
<p>New Features:</p>

In [34]:
player_stats = {}

WhiteGames = []
WhiteWin = []
WhiteLoss = []
WhiteDraw = []
BlackGames = []
BlackWin = []
BlackLoss = []
BlackDraw = []

WhiteWinRate = []
WhiteLossRate = []
WhiteDrawRate = []
BlackWinRate = []
BlackLossRate = []
BlackDrawRate = []

WLast10 = []
BLast10 = []

WasWGames = []
WasWWin = []
WasWLoss = []
WasWDraw = []
WasBGames = []
WasBWin = []
WasBLoss = []
WasBDraw = []
BasWGames = []
BasWWin = []
BasWLoss = []
BasWDraw = []
BasBGames = []
BasBWin = []
BasBLoss = []
BasBDraw = []

WasWWinRate = []
WasWLossRate = []
WasWDrawRate = []
WasBWinRate = []
WasBLossRate = []
WasBDrawRate = []
BasWWinRate = []
BasWLossRate = []
BasWDrawRate = []
BasBWinRate = []
BasBLossRate = []
BasBDrawRate = []

Storing the History:

In [35]:
def update_player(player, result, color):
    stats = player_stats.get(player)
    stats["games"] += 1
    if result == 1:
        if color == "white":
            stats["win"] += 1
            stats["whitegames"] += 1
            stats["whitewin"] += 1
            stats["last10"].append(result)
        else:
            stats["loss"] += 1
            stats["blackgames"] += 1
            stats["blackloss"] += 1
            stats["last10"].append(1 - result)
    elif result == 0.5:
        if color == "white":
            stats["draw"] += 1
            stats["whitegames"] += 1
            stats["whitedraw"] += 1
            stats["last10"].append(result)
        else:
            stats["draw"] += 1
            stats["blackgames"] += 1
            stats["blackdraw"] += 1
            stats["last10"].append(1 - result)
    else:
        if color == "white":
            stats["loss"] += 1
            stats["whitegames"] += 1
            stats["whiteloss"] += 1
            stats["last10"].append(result)
        else:
            stats["win"] += 1
            stats["blackgames"] += 1
            stats["blackwin"] += 1
            stats["last10"].append(1 - result)

In [36]:
for row in games.itertuples():
    white = row.White
    black = row.Black
    result = row.Result
    

    if white not in player_stats:
        player_stats[white] = {"games": 0, "win": 0, "loss": 0, "draw": 0, "whitegames": 0, "blackgames": 0, "whitewin": 0, "whiteloss": 0, "whitedraw": 0, "blackwin": 0, "blackloss": 0, "blackdraw": 0, "last10": deque([], maxlen=10)}
    if black not in player_stats:
        player_stats[black] = {"games": 0, "win": 0, "loss": 0, "draw": 0, "whitegames": 0, "blackgames": 0, "whitewin": 0, "whiteloss": 0, "whitedraw": 0, "blackwin": 0, "blackloss": 0, "blackdraw": 0, "last10": deque([], maxlen=10)}


    white_stats = player_stats.get(white)
    black_stats = player_stats.get(black)

    WhiteGames.append(white_stats["games"])
    WhiteWin.append(white_stats["win"])
    WhiteLoss.append(white_stats["loss"])
    WhiteDraw.append(white_stats["draw"])
    BlackGames.append(black_stats["games"])
    BlackWin.append(black_stats["win"])
    BlackLoss.append(black_stats["loss"])
    BlackDraw.append(black_stats["draw"])
    WasWGames.append(white_stats["whitegames"])
    WasWWin.append(white_stats["whitewin"])
    WasWLoss.append(white_stats["whiteloss"])
    WasWDraw.append(white_stats["whitedraw"])
    WasBGames.append(white_stats["blackgames"])
    WasBWin.append(white_stats["blackwin"])
    WasBLoss.append(white_stats["blackloss"])
    WasBDraw.append(white_stats["blackdraw"])
    BasWGames.append(black_stats["whitegames"])
    BasWWin.append(black_stats["whitewin"])
    BasWLoss.append(black_stats["whiteloss"])
    BasWDraw.append(black_stats["whitedraw"])
    BasBGames.append(black_stats["blackgames"])
    BasBWin.append(black_stats["blackwin"])
    BasBLoss.append(black_stats["blackloss"])
    BasBDraw.append(black_stats["blackdraw"])


    if white_stats["games"]:
        WhiteWinRate.append(white_stats["win"] / white_stats["games"])
        WhiteLossRate.append(white_stats["loss"] / white_stats["games"])
        WhiteDrawRate.append(white_stats["draw"] / white_stats["games"])
    else:
        WhiteWinRate.append(0)
        WhiteLossRate.append(0)
        WhiteDrawRate.append(0)
        
    if black_stats["games"]:
        BlackWinRate.append(black_stats["win"] / black_stats["games"])
        BlackLossRate.append(black_stats["loss"] / black_stats["games"])
        BlackDrawRate.append(black_stats["draw"] / black_stats["games"])
    else:
        BlackWinRate.append(0)
        BlackLossRate.append(0)
        BlackDrawRate.append(0)

    if white_stats["whitegames"]:
        WasWWinRate.append(white_stats["whitewin"] / white_stats["whitegames"])
        WasWLossRate.append(white_stats["whiteloss"] / white_stats["whitegames"])
        WasWDrawRate.append(white_stats["whitedraw"] / white_stats["whitegames"])
    else:
        WasWWinRate.append(0)
        WasWLossRate.append(0)
        WasWDrawRate.append(0)

    if white_stats["blackgames"]:
        WasBWinRate.append(white_stats["blackwin"] / white_stats["blackgames"])
        WasBLossRate.append(white_stats["blackloss"] / white_stats["blackgames"])
        WasBDrawRate.append(white_stats["blackdraw"] / white_stats["blackgames"])
    else:
        WasBWinRate.append(0)
        WasBLossRate.append(0)
        WasBDrawRate.append(0)

    if black_stats["whitegames"]:
        BasWWinRate.append(black_stats["whitewin"] / black_stats["whitegames"])
        BasWLossRate.append(black_stats["whiteloss"] / black_stats["whitegames"])
        BasWDrawRate.append(black_stats["whitedraw"] / black_stats["whitegames"])
    else:
        BasWWinRate.append(0)
        BasWLossRate.append(0)
        BasWDrawRate.append(0)

    if black_stats["blackgames"]:
        BasBWinRate.append(black_stats["blackwin"] / black_stats["blackgames"])
        BasBLossRate.append(black_stats["blackloss"] / black_stats["blackgames"])
        BasBDrawRate.append(black_stats["blackdraw"] / black_stats["blackgames"])
    else:
        BasBWinRate.append(0)
        BasBLossRate.append(0)
        BasBDrawRate.append(0)
    
    if white_stats["last10"]:
        WLast10.append(sum(white_stats["last10"]) / len(white_stats["last10"]))
    else:
        WLast10.append(0)
    if black_stats["last10"]:
        BLast10.append(sum(black_stats["last10"]) / len(black_stats["last10"]))
    else:
        BLast10.append(0)
    

    ###updating the features
    update_player(white, result, "white")
    update_player(black, result, "black")

Head2Head Statistics

New Features:

In [37]:
head2head = {}

h2h_games = []
h2h_Player1Win = []
h2h_Player2Win = []
h2h_Draw = []
h2h_Player1WinRate = []
h2h_Player2WinRate = []
h2h_Drawrate = []
h2h_last5 = []

Keeping track of H2H:

In [38]:
def update_h2h(pair, result, white, black):
    stats = head2head.get(pair)
    stats["games"] += 1
    stats["last5"].append(result)

    if result == 1:
        stats[white] += 1
    elif result == 0.5:
        stats["draw"] += 1
    else:
        stats[black] += 1

In [39]:
for row in games.itertuples():
    white = row.White
    black = row.Black
    result = row.Result
    pair = tuple(sorted((white, black)))

    if pair not in head2head:
        head2head[pair] = {"games": 0, pair[0]: 0, pair[1]: 0, "draw": 0, "last5": deque([], maxlen=5)}
    
    pair_stats = head2head[pair]

    h2h_games.append(pair_stats["games"])
    h2h_Player1Win.append(pair_stats[pair[0]])
    h2h_Player2Win.append(pair_stats[pair[1]])
    h2h_Draw.append(pair_stats["draw"])
    
    if pair_stats["games"]:
        h2h_Player1WinRate.append(pair_stats[pair[0]] / pair_stats["games"])
        h2h_Player2WinRate.append(pair_stats[pair[1]] / pair_stats["games"])
        h2h_Drawrate.append(pair_stats["draw"] / pair_stats["games"])
        h2h_last5.append(sum(pair_stats["last5"]) / len(pair_stats["last5"]))
    else:
        h2h_Player1WinRate.append(0)
        h2h_Player2WinRate.append(0)
        h2h_Drawrate.append(0)
        h2h_last5.append(0)

    ###updating the h2h features
    update_h2h(pair, result, white, black)

Date Features

In [40]:
Player_LastGameDate = {}

WhiteDaysSinceLastGame = []
BlackDaysSinceLastGame = []

In [41]:
for row in games.itertuples():
    white = row.White
    black = row.Black
    date = row.Date

    if white not in Player_LastGameDate:
        Player_LastGameDate[white] = date
    if black not in Player_LastGameDate:
        Player_LastGameDate[black] = date

    WhiteDaysSinceLastGame.append((date - Player_LastGameDate[white]).days)
    BlackDaysSinceLastGame.append((date - Player_LastGameDate[black]).days)

    ###update features
    Player_LastGameDate[white] = date
    Player_LastGameDate[black] = date

Diff 

New Features:

In [42]:
Last10Diff = []
GamesDiff = []
WinDiff = []
WinRateDiff = []
h2h_WinDiff = []
h2h_WinRateDiff = []
DaysSinceLastGameDiff = []

Calculating the Diff stats:

In [43]:
for i in range(len(games)):
    Last10Diff.append(WLast10[i] - BLast10[i])
    GamesDiff.append(WhiteGames[i] - BlackGames[i])
    WinDiff.append(WhiteWin[i] - BlackWin[i])
    WinRateDiff.append(WhiteWinRate[i] - BlackWinRate[i])
    h2h_WinDiff.append(h2h_Player1Win[i] - h2h_Player2Win[i])
    h2h_WinRateDiff.append(h2h_Player1WinRate[i] - h2h_Player2WinRate[i])
    DaysSinceLastGameDiff.append(WhiteDaysSinceLastGame[i] - BlackDaysSinceLastGame[i])

ECO Stats(opening stats)

New Features:

In [44]:
ECOs = {}

ECO_Games = []
ECO_WhiteWin = []
ECO_BlackWin = []
ECO_Draw = []
ECO_WhiteWinRate = []
ECO_BlackWinRate = []
ECO_DrawRate = []


Updating Eco:

In [45]:
def update_eco(eco, result):
    stats = ECOs[eco]
    stats["games"] += 1
    if result == 1:
        stats["whitewin"] += 1
    elif result == 0.5:
        stats["draw"] += 1
    else:
        stats["blackwin"] += 1

Storing the Stats:

In [46]:
for row in games.itertuples():
    result = row.Result
    eco = row.ECO

    if eco not in ECOs:
        ECOs[eco] = {"games": 0, "whitewin": 0, "blackwin": 0, "draw": 0}
    eco_stats = ECOs[eco]
    
    ECO_Games.append(eco_stats["games"])
    ECO_WhiteWin.append(eco_stats["whitewin"])
    ECO_BlackWin.append(eco_stats["blackwin"])
    ECO_Draw.append(eco_stats["draw"])
    if eco_stats["games"]:
        ECO_WhiteWinRate.append(eco_stats["whitewin"] / eco_stats["games"])
        ECO_BlackWinRate.append(eco_stats["blackwin"] / eco_stats["games"])
        ECO_DrawRate.append(eco_stats["draw"] / eco_stats["games"])
    else:
        ECO_WhiteWinRate.append(0)
        ECO_BlackWinRate.append(0)
        ECO_DrawRate.append(0)
    
    ###updating features
    update_eco(eco, result)

Adding all New Features to the "games" DataFrame

In [47]:
games["WhiteGames"] = WhiteGames
games["WhiteWin"] = WhiteWin
games["WhiteLoss"] = WhiteLoss
games["WhiteDraw"] = WhiteDraw
games["BlackGames"] = BlackGames
games["BlackWin"] = BlackWin
games["BlackLoss"] = BlackLoss
games["BlackDraw"] = BlackDraw

games["WhiteWinRate"] = WhiteWinRate
games["WhiteLossRate"] = WhiteLossRate
games["WhiteDrawRate"] = WhiteDrawRate
games["BlackWinRate"] = BlackWinRate
games["BlackLossRate"] = BlackLossRate
games["BlackDrawRate"] = BlackDrawRate

games["WLast10"] = WLast10
games["BLast10"] = BLast10

games["WasWGames"] = WasWGames
games["WasWWin"] = WasWWin
games["WasWLoss"] = WasWLoss
games["WasWDraw"] = WasWDraw
games["WasBGames"] = WasBGames
games["WasBWin"] = WasBWin
games["WasBLoss"] = WasBLoss
games["WasBDraw"] = WasBDraw

games["BasWGames"] = BasWGames
games["BasWWin"] = BasWWin
games["BasWLoss"] = BasWLoss
games["BasWDraw"] = BasWDraw
games["BasBGames"] = BasBGames
games["BasBWin"] = BasBWin
games["BasBLoss"] = BasBLoss
games["BasBDraw"] = BasBDraw

games["WasWWinRate"] = WasWWinRate
games["WasWLossRate"] = WasWLossRate
games["WasWDrawRate"] = WasWDrawRate
games["WasBWinRate"] = WasBWinRate
games["WasBLossRate"] = WasBLossRate
games["WasBDrawRate"] = WasBDrawRate

games["BasWWinRate"] = BasWWinRate
games["BasWLossRate"] = BasWLossRate
games["BasWDrawRate"] = BasWDrawRate
games["BasBWinRate"] = BasBWinRate
games["BasBLossRate"] = BasBLossRate
games["BasBDrawRate"] = BasBDrawRate

games["h2h_games"] = h2h_games
games["h2h_Player1Win"] = h2h_Player1Win
games["h2h_Player2Win"] = h2h_Player2Win
games["h2h_Draw"] = h2h_Draw
games["h2h_Player1WinRate"] = h2h_Player1WinRate
games["h2h_Player2WinRate"] = h2h_Player2WinRate
games["h2h_DrawRate"] = h2h_Drawrate
games["h2h_last5"] = h2h_last5

games["WhiteDaysSinceLastGame"] = WhiteDaysSinceLastGame
games["BlackDaysSinceLastGame"] = BlackDaysSinceLastGame

games["Last10Diff"] = Last10Diff
games["GamesDiff"] = GamesDiff
games["WinDiff"] = WinDiff
games["WinRateDiff"] = WinRateDiff
games["h2h_WinDiff"] = h2h_WinDiff
games["h2h_WinRateDiff"] = h2h_WinRateDiff
games["DaysSinceLastGameDiff"] = DaysSinceLastGameDiff

games["ECO_Games"] = ECO_Games
games["ECO_WhiteWin"] = ECO_WhiteWin
games["ECO_BlackWin"] = ECO_BlackWin
games["ECO_Draw"] = ECO_Draw
games["ECO_WhiteWinRate"] = ECO_WhiteWinRate
games["ECO_BlackWinRate"] = ECO_BlackWinRate
games["ECO_DrawRate"] = ECO_DrawRate

Checking the New Features

In [48]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 3255656 entries, 0 to 3255655
Data columns (total 79 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   Site                    str           
 1   Date                    datetime64[us]
 2   White                   str           
 3   Black                   str           
 4   Result                  float64       
 5   WhiteElo                float64       
 6   BlackElo                float64       
 7   ECO                     str           
 8   EloDiff                 float64       
 9   AvgElo                  float64       
 10  HigherRatedPlayer       object        
 11  WhiteGames              int64         
 12  WhiteWin                int64         
 13  WhiteLoss               int64         
 14  WhiteDraw               int64         
 15  BlackGames              int64         
 16  BlackWin                int64         
 17  BlackLoss               int64         
 18  BlackDraw    

In [49]:
games.columns

Index(['Site', 'Date', 'White', 'Black', 'Result', 'WhiteElo', 'BlackElo',
       'ECO', 'EloDiff', 'AvgElo', 'HigherRatedPlayer', 'WhiteGames',
       'WhiteWin', 'WhiteLoss', 'WhiteDraw', 'BlackGames', 'BlackWin',
       'BlackLoss', 'BlackDraw', 'WhiteWinRate', 'WhiteLossRate',
       'WhiteDrawRate', 'BlackWinRate', 'BlackLossRate', 'BlackDrawRate',
       'WLast10', 'BLast10', 'WasWGames', 'WasWWin', 'WasWLoss', 'WasWDraw',
       'WasBGames', 'WasBWin', 'WasBLoss', 'WasBDraw', 'BasWGames', 'BasWWin',
       'BasWLoss', 'BasWDraw', 'BasBGames', 'BasBWin', 'BasBLoss', 'BasBDraw',
       'WasWWinRate', 'WasWLossRate', 'WasWDrawRate', 'WasBWinRate',
       'WasBLossRate', 'WasBDrawRate', 'BasWWinRate', 'BasWLossRate',
       'BasWDrawRate', 'BasBWinRate', 'BasBLossRate', 'BasBDrawRate',
       'h2h_games', 'h2h_Player1Win', 'h2h_Player2Win', 'h2h_Draw',
       'h2h_Player1WinRate', 'h2h_Player2WinRate', 'h2h_DrawRate', 'h2h_last5',
       'WhiteDaysSinceLastGame', 'BlackDaysSinceLast

In [50]:
games.describe()

,Date,Result,WhiteElo,BlackElo,EloDiff,AvgElo,WhiteGames,WhiteWin,WhiteLoss,WhiteDraw,...,h2h_WinDiff,h2h_WinRateDiff,DaysSinceLastGameDiff,ECO_Games,ECO_WhiteWin,ECO_BlackWin,ECO_Draw,ECO_WhiteWinRate,ECO_BlackWinRate,ECO_DrawRate
count,3255656,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,...,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06,3.255656e+06
mean,2021-02-08 05:25:18.590907,5.346867e-01,2.246863e+03,2.243630e+03,3.232742e+00,2.245246e+03,4.850139e+02,2.200190e+02,1.532818e+02,1.117130e+02,...,2.402465e-02,1.469467e-03,-1.648310e-01,1.181017e+04,4.867711e+03,4.091593e+03,2.850863e+03,4.006885e-01,3.269762e-01,2.721814e-01
min,2012-06-02 00:00:00,0.000000e+00,1.001000e+03,1.001000e+03,-1.663000e+03,1.009000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,-5.100000e+01,-1.000000e+00,-5.037000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2018-03-25 00:00:00,0.000000e+00,2.092000e+03,2.089000e+03,-1.610000e+02,2.113000e+03,3.900000e+01,1.200000e+01,1.400000e+01,8.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,2.307000e+03,8.970000e+02,7.040000e+02,6.640000e+02,3.764530e-01,2.911651e-01,2.280263e-01
50%,2022-01-27 00:00:00,5.000000e-01,2.279000e+03,2.276000e+03,4.000000e+00,2.269000e+03,1.780000e+02,6.600000e+01,5.900000e+01,3.800000e+01,...,0.000000e+00,0.000000e+00,0.000000e+00,6.866000e+03,2.697000e+03,2.201000e+03,1.887000e+03,4.028386e-01,3.315209e-01,2.672114e-01
75%,2024-05-14 00:00:00,1.000000e+00,2.435000e+03,2.433000e+03,1.680000e+02,2.411000e+03,5.730000e+02,2.370000e+02,1.870000e+02,1.250000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,1.695700e+04,6.886000e+03,5.706000e+03,4.276000e+03,4.287916e-01,3.665263e-01,3.103865e-01
max,2026-06-08 00:00:00,1.000000e+00,2.882000e+03,2.882000e+03,1.663000e+03,2.850000e+03,9.124000e+03,5.199000e+03,2.797000e+03,2.550000e+03,...,7.000000e+01,1.000000e+00,5.012000e+03,8.449900e+04,3.607300e+04,3.373200e+04,1.469500e+04,1.000000e+00,1.000000e+00,1.000000e+00
std,NaN,4.371975e-01,2.737053e+02,2.745517e+02,2.485373e+02,2.443439e+02,7.913328e+02,4.089362e+02,2.346794e+02,1.969564e+02,...,1.434767e+00,3.449458e-01,1.710810e+02,1.312467e+04,5.573900e+03,4.893240e+03,2.816052e+03,4.861874e-02,5.885407e-02,6.492513e-02


In [51]:
games.to_csv(os.path.join(project_dir, "data/final_chess.csv"), index=False)